# Lab 1 : Custom tools and tool calling

*Week 5 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read what it prints before you run the next cell.


## What we are achieving in this lab

In Weeks 1 and 2 you asked a **language model** a question and it wrote an answer.(for example Claude).

In Weeks 3 and 4 you gave the model your company documents, so it could answer from those documents(RAG).

One important case is still missing. Imagine a customer asks: "Has order ORD-1007 shipped?"

The data for that order lives in **your application**  for example a system database, or a Python dictionary you maintain. The language model does **not** know that data. It was not trained on your live orders, and it cannot read your database by itself.

Therefore, if you send only the customer question to the model, and you do not give it a function that can look up ORD-1007 in your database (or wherever you store that data), the model cannot return the real order status. It may invent a status that sounds correct. That incorrect answer could then be shown to a customer.

In this week we will understand how models can take actions by calling tools.

Tools are nothing but the Python code written by you. A tool is a normal function. You tell the model that this function exists. The model does not run the function on its own. The model only asks for it: it says which function to use and which values to pass in.

That request is called a **tool call**. This lab focuses on tool calling: you will see the model choose different tools for different questions.



### Example we use in this lab

We use a small **company order desk** example. Support staff often need three lookups:

- Has this order shipped?
- Is this product in stock?
- What will shipping cost to this city?

In the next steps you will write these lookups as your own Python functions, turn them into tools, and then let the model ask for them.

### What you will do

1. Write three normal Python functions and run them yourself.
2. Turn those functions into tools the model can ask for.
3. Ask an order question and see the model choose `get_order_status`.
4. Ask a stock question and see the model choose `check_stock`.
5. Ask a shipping question and see the model choose `estimate_shipping`.
6. Ask a question with **no** tools, so you see the difference.

**Before you start.** Copy `.env.example` to `.env` in this `week05` folder. Paste your `ANTHROPIC_API_KEY`. Open this notebook from the `week05` folder. Setup steps are in [README.md](./README.md).

**Cost.** A few short Claude calls.


### Step 1. Load the API key

The next steps call Claude on the internet. Claude is the language model that will decide when to use your tools.

Do these three things first:

1. Make sure you have a file named `.env` in this folder.
2. Inside `.env`, put your key on a line like `ANTHROPIC_API_KEY=...`
3. Do not upload `.env` to git. The key is a secret.

The next cell runs `load_dotenv()`. That reads the `.env` file so the rest of the notebook can use your key.


In [2]:
from dotenv import load_dotenv

load_dotenv()  # reads .env from this folder (week05)
print("API key loaded from .env (if the file and key are present).")


API key loaded from .env (if the file and key are present).


### Step 2. Write custom Python functions (these are not tools yet)

In this step we only write **normal Python functions**. There is no language model here yet. There is no "tool" object yet.

Why start here?

Because a tool is not a new kind of magic. A tool is a function you already understand, plus a short description so the model knows when to ask for it.

We also make a small fake company dataset with dictionaries. In a real company this data would come from a database. Here it stays in Python so you can see every value.


In [3]:
# Tiny company data for an order desk. This is mock data for learning.
ORDERS = {
    "ORD-1007": {"customer": "John Paul", "item": "Laptop stand", "status": "shipped", "city": "Pune"},
    "ORD-1008": {"customer": "Alex Kim", "item": "USB-C hub", "status": "processing", "city": "Austin"},
    "ORD-1009": {"customer": "carmel", "item": "Webcam", "status": "delivered", "city": "London"},
}

STOCK = {
    "Laptop stand": 42,
    "USB-C hub": 5,
    "Webcam": 0,
    "HDMI cable": 120,
}

# Shipping rate: flat amount per city for this demo.
SHIPPING_RATE_USD = {
    "california": 4.5,
    "texas": 6.0,
    "newyork": 9.5,
}


def get_order_status(order_id: str) -> dict:
    """Return the status and details for one order id."""
    order = ORDERS.get(order_id)
    if order is None:
        return {"error": f"Order {order_id} was not found"}
    return {"order_id": order_id, **order}


def check_stock(product_name: str) -> dict:
    """Return how many units of a product are in stock."""
    if product_name not in STOCK:
        return {"error": f"Product {product_name} was not found"}
    quantity = STOCK[product_name]
    return {
        "product_name": product_name,
        "quantity_in_stock": quantity,
    }


def estimate_shipping(city: str, weight_kg: float) -> dict:
    """Estimate shipping cost in USD for a city and package weight."""
    if city not in SHIPPING_RATE_USD:
        return {"error": f"No shipping rate for city {city}"}
    cost = SHIPPING_RATE_USD[city] * weight_kg
    return {"city": city, "weight_kg": weight_kg, "shipping_cost_usd": cost}

print("Functions defined:")
print("- get_order_status")
print("- check_stock")
print("- estimate_shipping")


Functions defined:
- get_order_status
- check_stock
- estimate_shipping


### Step 3. Call the functions yourself

Now call each function the normal way, like any Python function.

This step matters. **You** can run these functions without a language model.

Later, when we say "the agent used a tool", it is still the **same function**. The new part is only this: the **model chooses** the function name and the input values. Your Python code still runs the function.


In [4]:

print(get_order_status("ORD-1007"))

{'order_id': 'ORD-1007', 'customer': 'John Paul', 'item': 'Laptop stand', 'status': 'shipped', 'city': 'Pune'}


In [5]:

print(check_stock("USB-C hub"))

{'product_name': 'USB-C hub', 'quantity_in_stock': 5}


In [6]:
print("Shipping estimate:")
print(estimate_shipping("texas", weight_kg=2.0))
print()



Shipping estimate:
{'city': 'texas', 'weight_kg': 2.0, 'shipping_cost_usd': 12.0}



In [7]:
print("Missing order (error path):")
print(get_order_status("ORD-9999"))

Missing order (error path):
{'error': 'Order ORD-9999 was not found'}


### Step 4. What is a tool?

A **tool** is your Python function, prepared so a language model can ask for it.

The model needs three pieces of information about each tool:

1. **Name** — for example `get_order_status`
2. **Description** — a short sentence that explains when to use it (this often comes from the function docstring)
3. **Arguments** — the inputs, for example `order_id`

Important: the model does **not** get your whole Python file to execute. It only gets a list of tool names and descriptions.

When the model wants data, it does not run the function. It sends back a request like:

- use tool: `get_order_status`
- with argument: `order_id = "ORD-1007"`

That request is called a **tool call**.

In LangChain, the `@tool` line turns your function into a tool object. The logic inside the function stays the same as in Step 2. We only wrap it so Claude can see the name and description.


In [8]:
from langchain_core.tools import tool


@tool
def get_order_status(order_id: str) -> dict:
    """Look up one customer order by order id. Returns customer, item, status, and city."""
    order = ORDERS.get(order_id)
    if order is None:
        return {"error": f"Order {order_id} was not found"}
    return {"order_id": order_id, **order}


@tool
def check_stock(product_name: str) -> dict:
    """Check how many units of a product are in stock in the warehouse."""
    if product_name not in STOCK:
        return {"error": f"Product {product_name} was not found"}
    quantity = STOCK[product_name]
    return {
        "product_name": product_name,
        "quantity_in_stock": quantity,
    }


@tool
def estimate_shipping(city: str, weight_kg: float) -> dict:
    """Estimate shipping cost in USD for a city and package weight."""
    if city not in SHIPPING_RATE_USD:
        return {"error": f"No shipping rate for city {city}"}
    cost = SHIPPING_RATE_USD[city] * weight_kg
    return {"city": city, "weight_kg": weight_kg, "shipping_cost_usd": cost}

# The agent will only be allowed to use tools that appear in this list.
TOOLS = [get_order_status, check_stock, estimate_shipping]

print("These are now LangChain tools:")
for t in TOOLS:
    print()
    print("name       :", t.name)
    print("description:", t.description)


These are now LangChain tools:

name       : get_order_status
description: Look up one customer order by order id. Returns customer, item, status, and city.

name       : check_stock
description: Check how many units of a product are in stock in the warehouse.

name       : estimate_shipping
description: Estimate shipping cost in USD for a city and package weight.


Look at the names and descriptions that were printed.

The language model will read those descriptions and decide **which** tool fits the user question.

You can still run a tool yourself with `.invoke(...)`. That is the same "run the function" step the agent will use later.


In [12]:
# Same functions, now as tools. Your code still runs them.
print(get_order_status.invoke({"order_id": "ORD-1008"}))
print(check_stock.invoke({"product_name": "Webcam"}))
print(estimate_shipping.invoke({"city": "texas", "weight_kg": 1.0}))


{'order_id': 'ORD-1008', 'customer': 'Alex Kim', 'item': 'USB-C hub', 'status': 'processing', 'city': 'Austin'}
{'product_name': 'Webcam', 'quantity_in_stock': 0}
{'city': 'texas', 'weight_kg': 1.0, 'shipping_cost_usd': 6.0}


### Step 5. How the language model decides which tool to use

Now we create the Claude model. Then we use `bind_tools(TOOLS)`. That tells the model: "these are the tools you are allowed to ask for."

When you send a question, two things can happen:

- The model needs company data → it returns a **tool call** (tool name + arguments)
- The model can answer without data → it returns normal text

You do **not** write rules like "if the question is about an order, call get_order_status". The model chooses from the list you gave it.

Your job in this course is to:

1. write clear functions,
2. give clear tool names and descriptions,
3. run whatever tool call the model returns,
4. send the result back to the model.

The next cell sends one question and prints the model's decision. We do **not** run the tool yet. We only look at which tool the model asked for.


In [13]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# Attach our custom tools so the model knows they exist.
llm_with_tools = llm.bind_tools(TOOLS)

decision = llm_with_tools.invoke(
    [
        SystemMessage(
            content=(
                "You are an order-desk assistant for a company. "
                "When you need order, stock, or shipping data, use a tool. "
                "Do not invent order ids or stock numbers."
            )
        ),
        HumanMessage(content="What is the status of order ORD-1007?"),
    ]
)


print(decision.content)
print()
print("Tool calls chosen by the model:")
print(decision.tool_calls)


[{'id': 'toolu_01FrM7WtBcLv4ABJ9xpEvx6o', 'caller': {'type': 'direct'}, 'input': {'order_id': 'ORD-1007'}, 'name': 'get_order_status', 'type': 'tool_use', 'toolset_name': None}]

Tool calls chosen by the model:
[{'name': 'get_order_status', 'args': {'order_id': 'ORD-1007'}, 'id': 'toolu_01FrM7WtBcLv4ABJ9xpEvx6o', 'type': 'tool_call'}]


You should see a tool call for `get_order_status`, with `order_id` set to `ORD-1007`.

That means tool calling worked: the model picked a tool from your list and filled in the arguments.

If `tool_calls` is empty, and the model invents an order status, stop and fix the tool setup before you continue.

### Step 6. Same idea, different question: stock

Now ask about stock. The tools list is the same. Only the question changes.

Watch `tool_calls`. The model should choose `check_stock`, not `get_order_status`.


In [14]:
decision_stock = llm_with_tools.invoke(
    [
        SystemMessage(
            content=(
                "You are an order-desk assistant for a company. "
                "When you need order, stock, or shipping data, use a tool. "
                "Do not invent stock numbers."
            )
        ),
        HumanMessage(content="How many USB-C hubs do we have in stock?"),
    ]
)

print("Model text:")
print(decision_stock.content)
print()
print("Tool calls chosen by the model:")
print(decision_stock.tool_calls)


Model text:
[{'id': 'toolu_01F9RrTG39FBBCYkdbagULgS', 'caller': {'type': 'direct'}, 'input': {'product_name': 'USB-C hubs'}, 'name': 'check_stock', 'type': 'tool_use', 'toolset_name': None}]

Tool calls chosen by the model:
[{'name': 'check_stock', 'args': {'product_name': 'USB-C hubs'}, 'id': 'toolu_01F9RrTG39FBBCYkdbagULgS', 'type': 'tool_call'}]


You should see `check_stock` with `product_name` set to something like `USB-C hub`.

Same tools. Different question. Different tool choice. That is how the model decides which tool to use.

### Step 7. Same idea again: shipping

Ask a shipping question. The model should choose `estimate_shipping`.


In [ ]:
decision_ship = llm_with_tools.invoke(
    [
        SystemMessage(
            content=(
                "You are an order-desk assistant for a company. "
                "When you need order, stock, or shipping data, use a tool. "
                "Do not invent shipping prices."
            )
        ),
        HumanMessage(
            content="What is the shipping cost to texas for a package that weighs 1 kg?"
        ),
    ]
)

print("Model text:")
print(decision_ship.content)
print()
print("Tool calls chosen by the model:")
print(decision_ship.tool_calls)


Model text:
[{'id': 'toolu_01Vv9VJKzLpQq8BgcJ8XeeEC', 'caller': {'type': 'direct'}, 'input': {'city': 'Austin', 'weight_kg': 1}, 'name': 'estimate_shipping', 'type': 'tool_use', 'toolset_name': None}]

Tool calls chosen by the model:
[{'name': 'estimate_shipping', 'args': {'city': 'Austin', 'weight_kg': 1}, 'id': 'toolu_01Vv9VJKzLpQq8BgcJ8XeeEC', 'type': 'tool_call'}]


You should see `estimate_shipping` with a city and a weight.

You have now seen three questions and three different tool choices:

| Question about | Tool the model should pick |
|----------------|----------------------------|
| order status | `get_order_status` |
| stock | `check_stock` |
| shipping cost | `estimate_shipping` |

### Step 8. Same question with no tools

If you remove the tools, the model cannot ask for your functions. It cannot read your `ORDERS` dictionary. It can only guess.

Compare this with Step 5, where the model asked for `get_order_status`.


In [16]:
no_tools = llm.invoke(
    [
        SystemMessage(content="Answer briefly."),
        HumanMessage(content="What is the status of order ORD-1007?"),
    ]
)

print("Model with NO tools:")
print(no_tools.content)
print()
print("There is no tool_calls here.")
print("Any status in this answer did not come from your ORDERS dictionary.")


Model with NO tools:
I don't have access to order management systems or databases, so I can't look up the status of order ORD-1007.

To find your order status, you could:

1. **Check your email** - Look for order confirmation or shipping notification emails
2. **Visit the company's website** - Log into your account and view order history
3. **Contact customer service** - Call or email the company directly with your order number
4. **Use a tracking link** - If you received a shipping notification, it may include a tracking number

Is there anything else I can help you with?

There is no tool_calls here.
Any status in this answer did not come from your ORDERS dictionary.


## What you should be able to explain



1. A **tool** is Python code you wrote. The model does not run it by itself. The model only **asks** for it.
2. A **tool call** is that ask: tool name + input values.
3. The model chooses a tool by reading the **name** and **description** you gave it.
4. The same tool list can serve many questions. The question decides which tool is chosen.
5. Without tools, the model can only guess about your company data.

**Lab 2** will show what happens next: your program **runs** the tool, sends the result back to the model, and repeats if needed. That full path is an **AI agent**. We keep that for the next lab so this lab stays only on tool calling.
